In [ ]:
!pip install pandas faker

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.0/2.0 MB 13.0 MB/s eta 0:00:00


In [ ]:
import pandas as pd
import numpy as np
import random
from faker import Faker
from pathlib import Path
from datetime import datetime, timedelta

fake = Faker("es_MX")
random.seed(42)
np.random.seed(42)

# =========================
# CONFIGURACIÓN
# =========================
SALIDA = Path("datos_fraude_grafos_2M_tx")
SALIDA.mkdir(exist_ok=True)

NUM_PERSONAS = 50_000
NUM_FRAUDE = 5_000
NUM_NORMALES = NUM_PERSONAS - NUM_FRAUDE

NUM_SOLICITUDES = 100_000
NUM_TRANSACCIONES = 2_000_000

NUM_DISPOSITIVOS = 80_000
NUM_IPS = 30_000

FRAUD_GROUP_SIZE = 25

# =========================
# CATÁLOGOS
# =========================
canales = ["web", "app", "sucursal"]
estatus_solicitud = ["aprobada", "rechazada", "revision"]
tipos_cuenta = ["debito", "credito_digital"]
estatus_cuenta = ["activa", "bloqueada", "suspendida"]
tipos_transaccion = ["compra", "transferencia", "retiro", "pago"]
estatus_transaccion = ["exitosa", "rechazada", "en_revision"]
tipos_documento = ["INE", "Pasaporte", "Licencia"]
tipos_dispositivo = ["Android", "iPhone", "Laptop", "Tablet"]
sistemas_operativos = ["Android", "iOS", "Windows", "Linux"]
proveedores = ["Telmex", "Izzi", "Totalplay", "AT&T", "Movistar"]

# =========================
# HELPERS
# =========================
def generar_curp():
    return "".join(random.choices("ABCDEFGHIJKLMNOPQRSTUVWXYZ0123456789", k=18))

def generar_rfc():
    return "".join(random.choices("ABCDEFGHIJKLMNOPQRSTUVWXYZ0123456789", k=13))

def generar_folio():
    return "".join(random.choices("ABCDEFGHIJKLMNOPQRSTUVWXYZ0123456789", k=12))

def fecha_aleatoria(inicio, fin):
    delta = fin - inicio
    return inicio + timedelta(days=random.randint(0, delta.days))

def fecha_np(inicio, fin, n):
    inicio_ts = int(inicio.timestamp())
    fin_ts = int(fin.timestamp())
    valores = np.random.randint(inicio_ts, fin_ts, size=n)
    return pd.to_datetime(valores, unit="s").strftime("%Y-%m-%dT%H:%M:%S")

# =========================
# 1. PERSONAS
# =========================
print("Generando personas...")

personas = []
fraud_ids = set(range(NUM_NORMALES + 1, NUM_PERSONAS + 1))

for i in range(1, NUM_PERSONAS + 1):
    nombre = fake.first_name()
    ap_pat = fake.last_name()
    ap_mat = fake.last_name()

    personas.append({
        "id_persona": f"P{i}",
        "nombre": nombre,
        "apellido_paterno": ap_pat,
        "apellido_materno": ap_mat,
        "fecha_nacimiento": fake.date_of_birth(minimum_age=18, maximum_age=70),
        "curp": generar_curp(),
        "rfc": generar_rfc(),
        "es_fraude": 1 if i in fraud_ids else 0
    })

df_personas = pd.DataFrame(personas)
df_personas.to_csv(SALIDA / "personas.csv", index=False)

# =========================
# 2. TELÉFONOS, CORREOS, DOMICILIOS, DOCUMENTOS
# =========================
print("Generando atributos personales...")

telefonos = []
correos = []
domicilios = []
documentos = []

persona_telefono = []
persona_correo = []
persona_domicilio = []
persona_documento = []

# Normales: atributos mayormente únicos
for i in range(1, NUM_NORMALES + 1):
    id_persona = f"P{i}"

    id_tel = f"T{i}"
    id_correo = f"C{i}"
    id_dom = f"DOM{i}"
    id_doc = f"DOC{i}"

    telefonos.append({"id_telefono": id_tel, "numero": f"55{random.randint(10000000,99999999)}"})
    correos.append({"id_correo": id_correo, "email": f"user{i}@mail.com"})
    domicilios.append({
        "id_domicilio": id_dom,
        "calle": fake.street_name(),
        "numero": random.randint(1, 999),
        "colonia": fake.city_suffix(),
        "ciudad": random.choice(["Ciudad de México", "Guadalajara", "Monterrey", "Puebla"]),
        "estado": random.choice(["CDMX", "Jalisco", "Nuevo León", "Puebla"]),
        "codigo_postal": random.randint(10000, 99999)
    })
    documentos.append({
        "id_documento": id_doc,
        "tipo_documento": random.choice(tipos_documento),
        "folio": generar_folio(),
        "estatus_validacion": random.choice(["valido", "revision", "rechazado"])
    })

    persona_telefono.append({"id_persona": id_persona, "id_telefono": id_tel})
    persona_correo.append({"id_persona": id_persona, "id_correo": id_correo})
    persona_domicilio.append({"id_persona": id_persona, "id_domicilio": id_dom})
    persona_documento.append({"id_persona": id_persona, "id_documento": id_doc})

# Fraude: grupos que comparten atributos
current = NUM_NORMALES + 1
group_id = 1

while current <= NUM_PERSONAS:
    shared_tel = f"T_FRAUD_{group_id}"
    shared_dom = f"DOM_FRAUD_{group_id}"

    telefonos.append({"id_telefono": shared_tel, "numero": f"55{random.randint(10000000,99999999)}"})
    domicilios.append({
        "id_domicilio": shared_dom,
        "calle": f"Calle Fraud {group_id}",
        "numero": random.randint(1, 999),
        "colonia": "Centro",
        "ciudad": "Ciudad de México",
        "estado": "CDMX",
        "codigo_postal": random.randint(10000, 99999)
    })

    for _ in range(FRAUD_GROUP_SIZE):
        if current > NUM_PERSONAS:
            break

        id_persona = f"P{current}"
        id_correo = f"C{current}"
        id_doc = f"DOC{current}"

        correos.append({"id_correo": id_correo, "email": f"synthetic{current}@mail.com"})
        documentos.append({
            "id_documento": id_doc,
            "tipo_documento": random.choice(tipos_documento),
            "folio": generar_folio(),
            "estatus_validacion": random.choice(["valido", "revision"])
        })

        persona_telefono.append({"id_persona": id_persona, "id_telefono": shared_tel})
        persona_correo.append({"id_persona": id_persona, "id_correo": id_correo})
        persona_domicilio.append({"id_persona": id_persona, "id_domicilio": shared_dom})
        persona_documento.append({"id_persona": id_persona, "id_documento": id_doc})

        current += 1

    group_id += 1

pd.DataFrame(telefonos).to_csv(SALIDA / "telefonos.csv", index=False)
pd.DataFrame(correos).to_csv(SALIDA / "correos.csv", index=False)
pd.DataFrame(domicilios).to_csv(SALIDA / "domicilios.csv", index=False)
pd.DataFrame(documentos).to_csv(SALIDA / "documentos.csv", index=False)

pd.DataFrame(persona_telefono).to_csv(SALIDA / "persona_telefono.csv", index=False)
pd.DataFrame(persona_correo).to_csv(SALIDA / "persona_correo.csv", index=False)
pd.DataFrame(persona_domicilio).to_csv(SALIDA / "persona_domicilio.csv", index=False)
pd.DataFrame(persona_documento).to_csv(SALIDA / "persona_documento.csv", index=False)

# =========================
# 3. DISPOSITIVOS E IPS
# =========================
print("Generando dispositivos e IPs...")

dispositivos = []
ips = []

for i in range(1, NUM_DISPOSITIVOS + 1):
    dispositivos.append({
        "id_dispositivo": f"D{i}",
        "tipo": random.choice(tipos_dispositivo),
        "sistema_operativo": random.choice(sistemas_operativos),
        "huella_digital": fake.sha1()
    })

for i in range(1, NUM_IPS + 1):
    ips.append({
        "id_ip": f"IP{i}",
        "direccion_ip": fake.ipv4_public(),
        "proveedor": random.choice(proveedores),
        "ubicacion_aproximada": random.choice(["CDMX", "Guadalajara", "Monterrey", "Puebla"])
    })

# Dispositivos/IPs especiales para fraude
num_fraud_groups = int(np.ceil(NUM_FRAUDE / FRAUD_GROUP_SIZE))

for g in range(1, num_fraud_groups + 1):
    dispositivos.append({
        "id_dispositivo": f"D_FRAUD_{g}",
        "tipo": random.choice(tipos_dispositivo),
        "sistema_operativo": random.choice(sistemas_operativos),
        "huella_digital": fake.sha1()
    })
    ips.append({
        "id_ip": f"IP_FRAUD_{g}",
        "direccion_ip": fake.ipv4_public(),
        "proveedor": random.choice(proveedores),
        "ubicacion_aproximada": "CDMX"
    })

pd.DataFrame(dispositivos).to_csv(SALIDA / "dispositivos.csv", index=False)
pd.DataFrame(ips).to_csv(SALIDA / "ips.csv", index=False)

# =========================
# 4. SOLICITUDES
# =========================
print("Generando solicitudes...")

solicitudes = []
persona_solicitud = []
solicitud_dispositivo = []
solicitud_ip = []

for i in range(1, NUM_SOLICITUDES + 1):
    # Personas fraudulentas tendrán más probabilidad de aparecer en solicitudes
    if random.random() < 0.20:
        persona_num = random.randint(NUM_NORMALES + 1, NUM_PERSONAS)
    else:
        persona_num = random.randint(1, NUM_NORMALES)

    id_persona = f"P{persona_num}"
    id_solicitud = f"S{i}"
    es_fraude = persona_num in fraud_ids

    solicitudes.append({
        "id_solicitud": id_solicitud,
        "fecha_solicitud": fecha_aleatoria(datetime(2025, 1, 1), datetime(2026, 4, 1)).date(),
        "canal": random.choice(["web", "app"] if es_fraude else canales),
        "estatus": random.choice(["aprobada", "revision"] if es_fraude else estatus_solicitud),
        "score_inicial": random.randint(500, 700) if es_fraude else random.randint(250, 700)
    })

    persona_solicitud.append({"id_persona": id_persona, "id_solicitud": id_solicitud})

    if es_fraude:
        group = ((persona_num - NUM_NORMALES - 1) // FRAUD_GROUP_SIZE) + 1
        id_disp = f"D_FRAUD_{group}"
        id_ip = f"IP_FRAUD_{group}"
    else:
        id_disp = f"D{random.randint(1, NUM_DISPOSITIVOS)}"
        id_ip = f"IP{random.randint(1, NUM_IPS)}"

    solicitud_dispositivo.append({"id_solicitud": id_solicitud, "id_dispositivo": id_disp})
    solicitud_ip.append({"id_solicitud": id_solicitud, "id_ip": id_ip})

pd.DataFrame(solicitudes).to_csv(SALIDA / "solicitudes.csv", index=False)
pd.DataFrame(persona_solicitud).to_csv(SALIDA / "persona_solicitud.csv", index=False)
pd.DataFrame(solicitud_dispositivo).to_csv(SALIDA / "solicitud_dispositivo.csv", index=False)
pd.DataFrame(solicitud_ip).to_csv(SALIDA / "solicitud_ip.csv", index=False)

# =========================
# 5. CUENTAS
# =========================
print("Generando cuentas...")

cuentas = []
persona_cuenta = []

for i in range(1, NUM_PERSONAS + 1):
    id_persona = f"P{i}"
    id_cuenta = f"CTA{i}"
    es_fraude = i in fraud_ids

    cuentas.append({
        "id_cuenta": id_cuenta,
        "fecha_apertura": fecha_aleatoria(datetime(2025, 1, 1), datetime(2026, 4, 1)).date(),
        "tipo_cuenta": random.choice(tipos_cuenta),
        "estatus": "activa" if es_fraude else random.choice(estatus_cuenta)
    })

    persona_cuenta.append({"id_persona": id_persona, "id_cuenta": id_cuenta})

pd.DataFrame(cuentas).to_csv(SALIDA / "cuentas.csv", index=False)
pd.DataFrame(persona_cuenta).to_csv(SALIDA / "persona_cuenta.csv", index=False)

# =========================
# 6. TRANSACCIONES MASIVAS
# =========================
print("Generando 2 millones de transacciones...")

cuentas_ids = np.array([f"CTA{i}" for i in range(1, NUM_PERSONAS + 1)])
fraud_cuentas = np.array([f"CTA{i}" for i in range(NUM_NORMALES + 1, NUM_PERSONAS + 1)])
normal_cuentas = np.array([f"CTA{i}" for i in range(1, NUM_NORMALES + 1)])

# 15% de transacciones asociadas a cuentas fraudulentas
fraud_mask = np.random.rand(NUM_TRANSACCIONES) < 0.15

cuentas_trans = np.empty(NUM_TRANSACCIONES, dtype=object)
cuentas_trans[fraud_mask] = np.random.choice(fraud_cuentas, size=fraud_mask.sum(), replace=True)
cuentas_trans[~fraud_mask] = np.random.choice(normal_cuentas, size=(~fraud_mask).sum(), replace=True)

df_transacciones = pd.DataFrame({
    "id_transaccion": [f"TXN{i}" for i in range(1, NUM_TRANSACCIONES + 1)],
    "fecha_hora": fecha_np(datetime(2025, 1, 1), datetime(2026, 4, 1), NUM_TRANSACCIONES),
    "monto": np.round(
        np.where(
            fraud_mask,
            np.random.lognormal(mean=8.5, sigma=1.0, size=NUM_TRANSACCIONES),
            np.random.lognormal(mean=7.0, sigma=0.9, size=NUM_TRANSACCIONES)
        ),
        2
    ),
    "tipo": np.random.choice(tipos_transaccion, size=NUM_TRANSACCIONES),
    "estatus": np.where(
        fraud_mask,
        np.random.choice(["exitosa", "en_revision"], size=NUM_TRANSACCIONES, p=[0.65, 0.35]),
        np.random.choice(estatus_transaccion, size=NUM_TRANSACCIONES, p=[0.85, 0.10, 0.05])
    )
})

df_cuenta_transaccion = pd.DataFrame({
    "id_cuenta": cuentas_trans,
    "id_transaccion": df_transacciones["id_transaccion"]
})

df_transacciones.to_csv(SALIDA / "transacciones.csv", index=False)
df_cuenta_transaccion.to_csv(SALIDA / "cuenta_transaccion.csv", index=False)

# =========================
# 7. RESUMEN
# =========================
print("\nProceso completado.")
print(f"Archivos guardados en: {SALIDA.resolve()}")
print(f"Personas: {NUM_PERSONAS:,}")
print(f"Fraude simulado: {NUM_FRAUDE:,}")
print(f"Solicitudes: {NUM_SOLICITUDES:,}")
print(f"Cuentas: {NUM_PERSONAS:,}")
print(f"Transacciones: {NUM_TRANSACCIONES:,}")

Generando personas...
Generando atributos personales...
Generando dispositivos e IPs...
Generando solicitudes...
Generando cuentas...
Generando 2 millones de transacciones...

Proceso completado.
Archivos guardados en: /content/datos_fraude_grafos_2M_tx
Personas: 50,000
Fraude simulado: 5,000
Solicitudes: 100,000
Cuentas: 50,000
Transacciones: 2,000,000


In [ ]:
import shutil

# =========================
# COMPRIMIR TODO EN ZIP
# =========================
zip_path = shutil.make_archive(
    "datos_fraude_grafos_2M_tx",
    "zip",
    SALIDA
)

print(f"\nZIP generado en: {zip_path}")


ZIP generado en: /content/datos_fraude_grafos_2M_tx.zip
